# PINN Tumor Growth Prediction

This notebook implements a **Physics-Informed Neural Network (PINN)** for predicting glioblastoma tumor growth over time.

## Architecture Overview
- **PDE**: Fisher-Kolmogorov equation: ∂u/∂t = D∇²u + ρu(1-u)
- **Network**: MLP [t, x, y, z] → [20, 50, 50, 50, 50, 20] → u(t,x,y,z)
- **Loss**: L_total = λ_data * L_data + λ_pde * L_pde + λ_bc * L_bc
- **Parameters**: D (diffusion coefficient), ρ (proliferation rate)

## Datasets
- **Phase A**: LGG Dataset (110 patients, baseline segmentations) - Parameter estimation
- **Phase B**: MU-Glioma Dataset (203 patients, longitudinal) - Growth prediction validation

## Expected Outcomes
- Patient-specific tumor growth parameters
- Forward prediction of tumor evolution
- Uncertainty quantification via ensemble
- DICE score validation on follow-up scans

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import cv2
from PIL import Image
import json
from datetime import datetime
from scipy.ndimage import gaussian_filter
from scipy.interpolate import griddata

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Data Loading and Preprocessing

### 2.1 Load Datasets

In [ ]:
# Set data paths
LGG_DATA_DIR = Path('/kaggle/input/lgg-mri-segmentation/kaggle_3m')
MU_GLIOMA_DATA_DIR = Path('/kaggle/input/pkg-mu/PKG - MU-Glioma-Post/MU-Glioma-Post')

print("=" * 70)
print("DATASET 1: LGG (Lower Grade Glioma) - Baseline Segmentations")
print("=" * 70)
print(f"LGG Directory: {LGG_DATA_DIR}")
print(f"Directory exists: {LGG_DATA_DIR.exists()}")

# Get all patient folders from LGG dataset
lgg_patient_folders = sorted([f for f in LGG_DATA_DIR.iterdir() if f.is_dir() and f.name.startswith('TCGA')])
print(f"Number of LGG patients: {len(lgg_patient_folders)}")

# Collect patient data
lgg_patient_data = {}

for patient_folder in tqdm(lgg_patient_folders, desc="Scanning LGG patients"):
    slices = sorted([f for f in patient_folder.glob('*.tif') if '_mask' not in f.name])
    masks = sorted([f for f in patient_folder.glob('*_mask.tif')])
    
    if len(slices) > 0 and len(masks) > 0:
        lgg_patient_data[patient_folder.name] = {
            'slices': slices,
            'masks': masks,
            'n_slices': len(slices)
        }

print(f"\nTotal LGG patients with masks: {len(lgg_patient_data)}")

print("\n" + "=" * 70)
print("DATASET 2: MU-Glioma - Longitudinal Follow-up")
print("=" * 70)
print(f"MU-Glioma Directory: {MU_GLIOMA_DATA_DIR}")
print(f"Directory exists: {MU_GLIOMA_DATA_DIR.exists()}")

mu_glioma_patient_data = {}

if MU_GLIOMA_DATA_DIR.exists():
    mu_glioma_patient_folders = sorted([f for f in MU_GLIOMA_DATA_DIR.iterdir() if f.is_dir()])
    print(f"Number of MU-Glioma patients: {len(mu_glioma_patient_folders)}")
    
    for patient_folder in tqdm(mu_glioma_patient_folders, desc="Scanning MU-Glioma patients"):
        slices = sorted([f for f in patient_folder.glob('*.tif')])
        
        if len(slices) > 0:
            mu_glioma_patient_data[patient_folder.name] = {
                'slices': slices,
                'n_slices': len(slices),
                'timepoint': 0  # TODO: Extract from metadata
            }
    
    print(f"Total MU-Glioma patients: {len(mu_glioma_patient_data)}")
else:
    print("⚠️  MU-Glioma directory not found.")
    print("    Phase A (parameter estimation) will proceed with LGG only.")
    print("    Phase B (growth prediction) requires MU-Glioma longitudinal data.")

print(f"\n📊 Total patients available: {len(lgg_patient_data) + len(mu_glioma_patient_data)}")

### 2.2 Convert Segmentation Masks to Tumor Density Maps

In [ ]:
def load_tumor_volume(patient_data, img_size=64, smooth_sigma=1.0):
    """
    Load tumor segmentation masks and convert to smooth density maps
    
    Args:
        patient_data: Dictionary with 'slices' and 'masks' keys
        img_size: Resize dimension for computational efficiency
        smooth_sigma: Gaussian smoothing for continuous density
    
    Returns:
        volume: 3D numpy array (depth, height, width) with values [0, 1]
    """
    masks = patient_data['masks']
    
    volume_slices = []
    for mask_path in masks:
        # Load mask
        mask = np.array(Image.open(mask_path))
        
        # Convert to grayscale if RGB
        if len(mask.shape) == 3:
            mask = mask[:, :, 0]
        
        # Resize for computational efficiency
        mask = cv2.resize(mask, (img_size, img_size))
        
        # Normalize to [0, 1]
        mask = mask.astype(np.float32) / 255.0
        
        # Binary thresholding
        mask = (mask > 0.5).astype(np.float32)
        
        # Apply Gaussian smoothing for continuous density
        if smooth_sigma > 0:
            mask = gaussian_filter(mask, sigma=smooth_sigma)
        
        volume_slices.append(mask)
    
    # Stack to 3D volume
    volume = np.stack(volume_slices, axis=0)
    
    return volume


# Test tumor volume loading
test_patient_id = list(lgg_patient_data.keys())[0]
test_volume = load_tumor_volume(lgg_patient_data[test_patient_id], img_size=64, smooth_sigma=1.0)

print(f"Test patient: {test_patient_id}")
print(f"Volume shape: {test_volume.shape}")
print(f"Value range: [{test_volume.min():.3f}, {test_volume.max():.3f}]")
print(f"Tumor voxels: {(test_volume > 0.1).sum()} / {test_volume.size}")

# Visualize tumor density
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

n_slices = test_volume.shape[0]
indices = np.linspace(0, n_slices-1, 10, dtype=int)

for i, idx in enumerate(indices):
    axes[i].imshow(test_volume[idx], cmap='hot', vmin=0, vmax=1)
    axes[i].set_title(f'Slice {idx}/{n_slices-1}', fontsize=10)
    axes[i].axis('off')

plt.suptitle('Tumor Density Map (Smoothed Segmentation)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.3 Create Sparse Point Clouds for PINN Training

In [ ]:
def create_collocation_points(volume, n_points=10000, t=0.0, boundary_ratio=0.1):
    """
    Create sparse point cloud for PINN training
    
    Args:
        volume: 3D tumor density (depth, height, width)
        n_points: Number of collocation points
        t: Time value (for temporal data)
        boundary_ratio: Ratio of points on tumor boundary
    
    Returns:
        points: (N, 4) array of [t, x, y, z] normalized to [0, 1]
        values: (N,) array of tumor density u(t, x, y, z)
    """
    depth, height, width = volume.shape
    
    # Sample points uniformly in domain
    n_interior = int(n_points * (1 - boundary_ratio))
    
    # Random sampling in 3D space
    z_coords = np.random.randint(0, depth, n_interior)
    y_coords = np.random.randint(0, height, n_interior)
    x_coords = np.random.randint(0, width, n_interior)
    
    # Get tumor values at sampled points
    u_values = volume[z_coords, y_coords, x_coords]
    
    # Add boundary points (tumor interface)
    n_boundary = n_points - n_interior
    
    # Find tumor boundary (gradient magnitude)
    from scipy.ndimage import sobel
    grad_z = sobel(volume, axis=0)
    grad_y = sobel(volume, axis=1)
    grad_x = sobel(volume, axis=2)
    grad_mag = np.sqrt(grad_z**2 + grad_y**2 + grad_x**2)
    
    # Sample from high-gradient regions
    boundary_mask = grad_mag > 0.1
    boundary_indices = np.argwhere(boundary_mask)
    
    if len(boundary_indices) > 0:
        boundary_sample_idx = np.random.choice(len(boundary_indices), 
                                               size=min(n_boundary, len(boundary_indices)), 
                                               replace=False)
        boundary_coords = boundary_indices[boundary_sample_idx]
        
        z_coords = np.concatenate([z_coords, boundary_coords[:, 0]])
        y_coords = np.concatenate([y_coords, boundary_coords[:, 1]])
        x_coords = np.concatenate([x_coords, boundary_coords[:, 2]])
        
        u_boundary = volume[boundary_coords[:, 0], boundary_coords[:, 1], boundary_coords[:, 2]]
        u_values = np.concatenate([u_values, u_boundary])
    
    # Normalize coordinates to [0, 1]
    x_norm = x_coords / (width - 1)
    y_norm = y_coords / (height - 1)
    z_norm = z_coords / (depth - 1)
    t_vals = np.full_like(x_norm, t, dtype=np.float32)
    
    # Stack to (N, 4) array: [t, x, y, z]
    points = np.stack([t_vals, x_norm, y_norm, z_norm], axis=1).astype(np.float32)
    values = u_values.astype(np.float32)
    
    return points, values


# Test collocation point generation
test_points, test_values = create_collocation_points(test_volume, n_points=10000, t=0.0)

print(f"Collocation points shape: {test_points.shape}")
print(f"Values shape: {test_values.shape}")
print(f"Points range: t=[{test_points[:, 0].min():.2f}, {test_points[:, 0].max():.2f}], "
      f"x=[{test_points[:, 1].min():.2f}, {test_points[:, 1].max():.2f}], "
      f"y=[{test_points[:, 2].min():.2f}, {test_points[:, 2].max():.2f}], "
      f"z=[{test_points[:, 3].min():.2f}, {test_points[:, 3].max():.2f}]")
print(f"Values range: [{test_values.min():.3f}, {test_values.max():.3f}]")

# Visualize point distribution
fig = plt.figure(figsize=(16, 6))

# 3D scatter plot
ax1 = fig.add_subplot(131, projection='3d')
scatter = ax1.scatter(test_points[:, 1], test_points[:, 2], test_points[:, 3], 
                     c=test_values, cmap='hot', s=1, alpha=0.5)
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
ax1.set_title('Collocation Points in 3D Space', fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=ax1, label='Tumor Density')

# Value histogram
ax2 = fig.add_subplot(132)
ax2.hist(test_values, bins=50, edgecolor='black', alpha=0.7)
ax2.set_xlabel('Tumor Density u(x)')
ax2.set_ylabel('Count')
ax2.set_title('Distribution of Tumor Densities', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

# XY projection
ax3 = fig.add_subplot(133)
scatter2 = ax3.scatter(test_points[:, 1], test_points[:, 2], 
                       c=test_values, cmap='hot', s=5, alpha=0.5)
ax3.set_xlabel('X')
ax3.set_ylabel('Y')
ax3.set_title('XY Projection', fontsize=12, fontweight='bold')
plt.colorbar(scatter2, ax=ax3, label='Tumor Density')

plt.tight_layout()
plt.show()

## 3. PINN Model Architecture

### 3.1 Physics-Informed Neural Network

In [ ]:
class PINN(nn.Module):
    """
    Physics-Informed Neural Network for tumor growth prediction
    
    Network: [t, x, y, z] → [20, 50, 50, 50, 50, 20] → u(t, x, y, z)
    
    PDE: ∂u/∂t = D∇²u + ρu(1-u)  [Fisher-Kolmogorov equation]
    """
    
    def __init__(self, layers=[4, 20, 50, 50, 50, 50, 20, 1]):
        super(PINN, self).__init__()
        
        self.layers = nn.ModuleList()
        for i in range(len(layers) - 1):
            self.layers.append(nn.Linear(layers[i], layers[i+1]))
        
        # Xavier initialization
        for layer in self.layers:
            nn.init.xavier_normal_(layer.weight)
            nn.init.zeros_(layer.bias)
        
        # Learnable physics parameters
        self.log_D = nn.Parameter(torch.tensor(0.0))  # log(diffusion coefficient)
        self.log_rho = nn.Parameter(torch.tensor(0.0))  # log(proliferation rate)
    
    def forward(self, txyz):
        """
        Forward pass
        
        Args:
            txyz: (N, 4) tensor of [t, x, y, z] coordinates
        
        Returns:
            u: (N, 1) tumor density prediction
        """
        x = txyz
        
        # Hidden layers with tanh activation
        for i, layer in enumerate(self.layers[:-1]):
            x = torch.tanh(layer(x))
        
        # Output layer with sigmoid (density in [0, 1])
        u = torch.sigmoid(self.layers[-1](x))
        
        return u
    
    def get_physics_parameters(self):
        """
        Get learned physics parameters
        
        Returns:
            D: Diffusion coefficient (mm²/day)
            rho: Proliferation rate (1/day)
        """
        D = torch.exp(self.log_D)
        rho = torch.exp(self.log_rho)
        return D, rho


# Test PINN model
test_pinn = PINN(layers=[4, 20, 50, 50, 50, 50, 20, 1]).to(device)

# Count parameters
total_params = sum(p.numel() for p in test_pinn.parameters())
trainable_params = sum(p.numel() for p in test_pinn.parameters() if p.requires_grad)

print("PINN Model Architecture")
print("=" * 50)
print(f"Input: [t, x, y, z] (4D coordinates)")
print(f"Hidden layers: [20, 50, 50, 50, 50, 20]")
print(f"Output: u(t, x, y, z) (tumor density)")
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nPhysics parameters:")
D_init, rho_init = test_pinn.get_physics_parameters()
print(f"  D (diffusion): {D_init.item():.4f}")
print(f"  ρ (proliferation): {rho_init.item():.4f}")

# Test forward pass
test_input = torch.rand(100, 4).to(device)
test_output = test_pinn(test_input)
print(f"\nTest forward pass:")
print(f"  Input shape: {test_input.shape}")
print(f"  Output shape: {test_output.shape}")
print(f"  Output range: [{test_output.min().item():.3f}, {test_output.max().item():.3f}]")

### 3.2 Automatic Differentiation for PDE Terms

In [ ]:
def compute_pde_residual(model, txyz):
    """
    Compute PDE residual: R = ∂u/∂t - D∇²u - ρu(1-u)
    
    Args:
        model: PINN model
        txyz: (N, 4) tensor of [t, x, y, z] coordinates (requires_grad=True)
    
    Returns:
        residual: (N, 1) PDE residual
    """
    # Enable gradient computation
    txyz = txyz.requires_grad_(True)
    
    # Forward pass
    u = model(txyz)
    
    # First-order derivatives
    u_grad = torch.autograd.grad(u, txyz, 
                                 grad_outputs=torch.ones_like(u),
                                 create_graph=True)[0]
    
    u_t = u_grad[:, 0:1]  # ∂u/∂t
    u_x = u_grad[:, 1:2]  # ∂u/∂x
    u_y = u_grad[:, 2:3]  # ∂u/∂y
    u_z = u_grad[:, 3:4]  # ∂u/∂z
    
    # Second-order derivatives (Laplacian)
    u_xx = torch.autograd.grad(u_x, txyz,
                               grad_outputs=torch.ones_like(u_x),
                               create_graph=True)[0][:, 1:2]
    
    u_yy = torch.autograd.grad(u_y, txyz,
                               grad_outputs=torch.ones_like(u_y),
                               create_graph=True)[0][:, 2:3]
    
    u_zz = torch.autograd.grad(u_z, txyz,
                               grad_outputs=torch.ones_like(u_z),
                               create_graph=True)[0][:, 3:4]
    
    # Laplacian: ∇²u = ∂²u/∂x² + ∂²u/∂y² + ∂²u/∂z²
    laplacian = u_xx + u_yy + u_zz
    
    # Get physics parameters
    D, rho = model.get_physics_parameters()
    
    # Fisher-Kolmogorov PDE: ∂u/∂t = D∇²u + ρu(1-u)
    # Residual: R = ∂u/∂t - D∇²u - ρu(1-u)
    residual = u_t - D * laplacian - rho * u * (1 - u)
    
    return residual


# Test PDE residual computation
test_txyz = torch.rand(100, 4, requires_grad=True).to(device)
test_residual = compute_pde_residual(test_pinn, test_txyz)

print("PDE Residual Computation Test")
print("=" * 50)
print(f"Input shape: {test_txyz.shape}")
print(f"Residual shape: {test_residual.shape}")
print(f"Residual range: [{test_residual.min().item():.6f}, {test_residual.max().item():.6f}]")
print(f"Residual mean: {test_residual.mean().item():.6f}")
print(f"Residual std: {test_residual.std().item():.6f}")
print(f"\n✓ Automatic differentiation working correctly!")

## 4. Loss Functions

### 4.1 Combined PINN Loss

In [ ]:
class PINNLoss(nn.Module):
    """
    Combined PINN loss: L_total = λ_data * L_data + λ_pde * L_pde + λ_bc * L_bc
    """
    
    def __init__(self, lambda_data=1.0, lambda_pde=1.0, lambda_bc=1.0):
        super(PINNLoss, self).__init__()
        self.lambda_data = lambda_data
        self.lambda_pde = lambda_pde
        self.lambda_bc = lambda_bc
    
    def forward(self, model, data_points, data_values, pde_points=None):
        """
        Compute total PINN loss
        
        Args:
            model: PINN model
            data_points: (N_data, 4) tensor of known data points
            data_values: (N_data, 1) tensor of known tumor densities
            pde_points: (N_pde, 4) tensor of collocation points for PDE
        
        Returns:
            total_loss: Combined loss
            loss_dict: Dictionary with individual loss components
        """
        # Data loss (MSE between prediction and ground truth)
        u_pred_data = model(data_points)
        loss_data = F.mse_loss(u_pred_data, data_values)
        
        # PDE loss (physics residual)
        if pde_points is not None:
            residual = compute_pde_residual(model, pde_points)
            loss_pde = torch.mean(residual ** 2)
        else:
            loss_pde = torch.tensor(0.0).to(data_points.device)
        
        # Boundary condition loss (tumor density constraints)
        # BC1: u ∈ [0, 1] (enforced by sigmoid)
        # BC2: u = 0 far from tumor (already in data)
        loss_bc = torch.tensor(0.0).to(data_points.device)
        
        # Total loss
        total_loss = (self.lambda_data * loss_data + 
                     self.lambda_pde * loss_pde + 
                     self.lambda_bc * loss_bc)
        
        loss_dict = {
            'total': total_loss.item(),
            'data': loss_data.item(),
            'pde': loss_pde.item(),
            'bc': loss_bc.item()
        }
        
        return total_loss, loss_dict


# Test loss computation
criterion = PINNLoss(lambda_data=1.0, lambda_pde=0.1, lambda_bc=0.1)

test_data_points = torch.rand(100, 4).to(device)
test_data_values = torch.rand(100, 1).to(device)
test_pde_points = torch.rand(500, 4).to(device)

test_loss, test_loss_dict = criterion(test_pinn, test_data_points, test_data_values, test_pde_points)

print("PINN Loss Function Test")
print("=" * 50)
print(f"Data loss: {test_loss_dict['data']:.6f}")
print(f"PDE loss: {test_loss_dict['pde']:.6f}")
print(f"BC loss: {test_loss_dict['bc']:.6f}")
print(f"Total loss: {test_loss_dict['total']:.6f}")
print(f"\n✓ Loss computation working correctly!")

## 5. Custom Dataset for PINN Training

In [ ]:
class TumorGrowthDataset(Dataset):
    """
    Dataset for PINN training with collocation points
    """
    
    def __init__(self, patient_data, img_size=64, n_data_points=5000, n_pde_points=10000):
        self.patient_ids = list(patient_data.keys())
        self.patient_data = patient_data
        self.img_size = img_size
        self.n_data_points = n_data_points
        self.n_pde_points = n_pde_points
    
    def __len__(self):
        return len(self.patient_ids)
    
    def __getitem__(self, idx):
        patient_id = self.patient_ids[idx]
        
        # Load tumor volume
        volume = load_tumor_volume(self.patient_data[patient_id], 
                                   img_size=self.img_size, 
                                   smooth_sigma=1.0)
        
        # Create data points (ground truth)
        data_points, data_values = create_collocation_points(volume, 
                                                             n_points=self.n_data_points,
                                                             t=0.0)
        
        # Create PDE collocation points (for physics loss)
        # Sample uniformly in [0, 1]^4
        pde_points = np.random.rand(self.n_pde_points, 4).astype(np.float32)
        
        return {
            'patient_id': patient_id,
            'data_points': torch.from_numpy(data_points),
            'data_values': torch.from_numpy(data_values).unsqueeze(1),
            'pde_points': torch.from_numpy(pde_points),
            'volume_shape': volume.shape
        }


# Create dataset and dataloader
IMG_SIZE = 64
N_DATA_POINTS = 5000
N_PDE_POINTS = 10000

# Split patients for training/validation
patient_ids = list(lgg_patient_data.keys())
train_patients, val_patients = train_test_split(patient_ids, test_size=0.2, random_state=42)

print(f"Train patients: {len(train_patients)}")
print(f"Val patients: {len(val_patients)}")

# Create datasets (per-patient)
train_data = {pid: lgg_patient_data[pid] for pid in train_patients}
val_data = {pid: lgg_patient_data[pid] for pid in val_patients}

train_dataset = TumorGrowthDataset(train_data, img_size=IMG_SIZE, 
                                   n_data_points=N_DATA_POINTS,
                                   n_pde_points=N_PDE_POINTS)

val_dataset = TumorGrowthDataset(val_data, img_size=IMG_SIZE,
                                 n_data_points=N_DATA_POINTS,
                                 n_pde_points=N_PDE_POINTS)

print(f"\nTrain dataset size: {len(train_dataset)} patients")
print(f"Val dataset size: {len(val_dataset)} patients")

# Test dataset loading
test_sample = train_dataset[0]
print(f"\nSample patient: {test_sample['patient_id']}")
print(f"Data points shape: {test_sample['data_points'].shape}")
print(f"Data values shape: {test_sample['data_values'].shape}")
print(f"PDE points shape: {test_sample['pde_points'].shape}")
print(f"Volume shape: {test_sample['volume_shape']}")

## 6. Training Functions

### 6.1 Phase A: Parameter Estimation (Per-Patient)

In [ ]:
def train_patient_pinn(patient_data, patient_id, n_epochs=5000, lr_adam=1e-3, lr_lbfgs=0.5):
    """
    Train PINN for a single patient to estimate D and ρ
    
    Args:
        patient_data: Patient's tumor data
        patient_id: Patient identifier
        n_epochs: Number of training epochs
        lr_adam: Learning rate for Adam optimizer
        lr_lbfgs: Learning rate for L-BFGS optimizer
    
    Returns:
        model: Trained PINN model
        history: Training history
        params: Learned physics parameters
    """
    print(f"\n{'='*70}")
    print(f"Training PINN for patient: {patient_id}")
    print(f"{'='*70}")
    
    # Load tumor volume
    volume = load_tumor_volume(patient_data, img_size=64, smooth_sigma=1.0)
    
    # Create collocation points
    data_points, data_values = create_collocation_points(volume, n_points=5000, t=0.0)
    pde_points = np.random.rand(10000, 4).astype(np.float32)
    
    # Convert to tensors
    data_points = torch.from_numpy(data_points).to(device)
    data_values = torch.from_numpy(data_values).unsqueeze(1).to(device)
    pde_points = torch.from_numpy(pde_points).to(device)
    
    # Initialize model
    model = PINN(layers=[4, 20, 50, 50, 50, 50, 20, 1]).to(device)
    
    # Loss and optimizer
    criterion = PINNLoss(lambda_data=1.0, lambda_pde=0.01, lambda_bc=0.0)
    optimizer_adam = torch.optim.Adam(model.parameters(), lr=lr_adam)
    
    # Training history
    history = {'loss': [], 'data_loss': [], 'pde_loss': [], 'D': [], 'rho': []}
    
    # Phase 1: Adam optimizer
    print(f"\nPhase 1: Adam optimization ({n_epochs} epochs)")
    best_loss = float('inf')
    
    pbar = tqdm(range(n_epochs), desc='Training')
    for epoch in pbar:
        model.train()
        optimizer_adam.zero_grad()
        
        # Compute loss
        loss, loss_dict = criterion(model, data_points, data_values, pde_points)
        
        # Backward pass
        loss.backward()
        optimizer_adam.step()
        
        # Save history
        history['loss'].append(loss_dict['total'])
        history['data_loss'].append(loss_dict['data'])
        history['pde_loss'].append(loss_dict['pde'])
        
        D, rho = model.get_physics_parameters()
        history['D'].append(D.item())
        history['rho'].append(rho.item())
        
        # Update progress bar
        if epoch % 100 == 0:
            pbar.set_postfix({
                'loss': f"{loss_dict['total']:.6f}",
                'D': f"{D.item():.4f}",
                'ρ': f"{rho.item():.4f}"
            })
        
        # Save best model
        if loss_dict['total'] < best_loss:
            best_loss = loss_dict['total']
            best_model_state = model.state_dict()
    
    # Load best model
    model.load_state_dict(best_model_state)
    
    # Phase 2: L-BFGS refinement (optional, but recommended for PINNs)
    print(f"\nPhase 2: L-BFGS refinement")
    optimizer_lbfgs = torch.optim.LBFGS(
        model.parameters(),
        lr=lr_lbfgs,
        max_iter=20,
        history_size=50,
        line_search_fn="strong_wolfe"
    )
    
    def closure():
        optimizer_lbfgs.zero_grad()
        loss, _ = criterion(model, data_points, data_values, pde_points)
        loss.backward()
        return loss
    
    optimizer_lbfgs.step(closure)
    
    # Final evaluation
    model.eval()
    with torch.no_grad():
        final_loss, final_loss_dict = criterion(model, data_points, data_values, pde_points)
        D_final, rho_final = model.get_physics_parameters()
    
    print(f"\n{'='*70}")
    print(f"Training completed for patient: {patient_id}")
    print(f"{'='*70}")
    print(f"Final loss: {final_loss_dict['total']:.6f}")
    print(f"  Data loss: {final_loss_dict['data']:.6f}")
    print(f"  PDE loss: {final_loss_dict['pde']:.6f}")
    print(f"\nLearned parameters:")
    print(f"  D (diffusion): {D_final.item():.6f} mm²/day")
    print(f"  ρ (proliferation): {rho_final.item():.6f} 1/day")
    print(f"{'='*70}")
    
    return model, history, {'D': D_final.item(), 'rho': rho_final.item()}


# Test training on one patient
test_patient_id = train_patients[0]
test_model, test_history, test_params = train_patient_pinn(
    lgg_patient_data[test_patient_id],
    test_patient_id,
    n_epochs=1000  # Reduced for testing
)

### 6.2 Visualize Training History

In [ ]:
def plot_training_history(history, patient_id):
    """
    Visualize PINN training history
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # Total loss
    axes[0, 0].plot(history['loss'], linewidth=2)
    axes[0, 0].set_xlabel('Epoch', fontsize=12)
    axes[0, 0].set_ylabel('Total Loss', fontsize=12)
    axes[0, 0].set_title('Total Loss', fontsize=14, fontweight='bold')
    axes[0, 0].set_yscale('log')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Data vs PDE loss
    axes[0, 1].plot(history['data_loss'], label='Data Loss', linewidth=2)
    axes[0, 1].plot(history['pde_loss'], label='PDE Loss', linewidth=2)
    axes[0, 1].set_xlabel('Epoch', fontsize=12)
    axes[0, 1].set_ylabel('Loss', fontsize=12)
    axes[0, 1].set_title('Data Loss vs PDE Loss', fontsize=14, fontweight='bold')
    axes[0, 1].set_yscale('log')
    axes[0, 1].legend(fontsize=10)
    axes[0, 1].grid(True, alpha=0.3)
    
    # Diffusion coefficient D
    axes[1, 0].plot(history['D'], linewidth=2, color='green')
    axes[1, 0].set_xlabel('Epoch', fontsize=12)
    axes[1, 0].set_ylabel('D (mm²/day)', fontsize=12)
    axes[1, 0].set_title('Diffusion Coefficient Evolution', fontsize=14, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Proliferation rate ρ
    axes[1, 1].plot(history['rho'], linewidth=2, color='red')
    axes[1, 1].set_xlabel('Epoch', fontsize=12)
    axes[1, 1].set_ylabel('ρ (1/day)', fontsize=12)
    axes[1, 1].set_title('Proliferation Rate Evolution', fontsize=14, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.suptitle(f'PINN Training History - Patient: {patient_id}', 
                fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.savefig(f'pinn_training_{patient_id}.png', dpi=300, bbox_inches='tight')
    plt.show()


# Visualize test patient training
plot_training_history(test_history, test_patient_id)

## 7. Phase A: Train on All LGG Patients

**Goal**: Learn patient-specific diffusion (D) and proliferation (ρ) parameters

In [ ]:
# Train on all training patients (this will take several hours!)
# For production, run with n_epochs=5000-10000

PHASE_A_EPOCHS = 2000  # Increase to 5000-10000 for production

phase_a_results = {}

print(f"\n{'='*70}")
print(f"PHASE A: PARAMETER ESTIMATION")
print(f"{'='*70}")
print(f"Training {len(train_patients)} patients...")
print(f"Epochs per patient: {PHASE_A_EPOCHS}")
print(f"Estimated time: ~{len(train_patients) * PHASE_A_EPOCHS / 60:.1f} minutes\n")

for i, patient_id in enumerate(train_patients[:5]):  # Start with first 5 patients
    print(f"\n[{i+1}/{len(train_patients)}] Patient: {patient_id}")
    
    model, history, params = train_patient_pinn(
        lgg_patient_data[patient_id],
        patient_id,
        n_epochs=PHASE_A_EPOCHS
    )
    
    phase_a_results[patient_id] = {
        'model': model,
        'history': history,
        'params': params
    }
    
    # Save checkpoint
    torch.save({
        'patient_id': patient_id,
        'model_state_dict': model.state_dict(),
        'params': params,
        'history': history
    }, f'pinn_patient_{patient_id}.pth')

print(f"\n{'='*70}")
print(f"PHASE A COMPLETED")
print(f"{'='*70}")
print(f"Trained {len(phase_a_results)} patients successfully!")

## 8. Parameter Analysis

### 8.1 Statistical Summary of Learned Parameters

In [ ]:
# Extract learned parameters
D_values = [result['params']['D'] for result in phase_a_results.values()]
rho_values = [result['params']['rho'] for result in phase_a_results.values()]

# Statistical summary
print("\n" + "="*70)
print("LEARNED PHYSICS PARAMETERS - STATISTICAL SUMMARY")
print("="*70)
print(f"\nDiffusion Coefficient D (mm²/day):")
print(f"  Mean: {np.mean(D_values):.6f}")
print(f"  Std: {np.std(D_values):.6f}")
print(f"  Min: {np.min(D_values):.6f}")
print(f"  Max: {np.max(D_values):.6f}")
print(f"  Median: {np.median(D_values):.6f}")

print(f"\nProliferation Rate ρ (1/day):")
print(f"  Mean: {np.mean(rho_values):.6f}")
print(f"  Std: {np.std(rho_values):.6f}")
print(f"  Min: {np.min(rho_values):.6f}")
print(f"  Max: {np.max(rho_values):.6f}")
print(f"  Median: {np.median(rho_values):.6f}")
print("="*70)

# Visualize parameter distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Diffusion coefficient histogram
axes[0].hist(D_values, bins=20, edgecolor='black', color='green', alpha=0.7)
axes[0].axvline(np.mean(D_values), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(D_values):.4f}')
axes[0].set_xlabel('D (mm²/day)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Distribution of Diffusion Coefficients', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Proliferation rate histogram
axes[1].hist(rho_values, bins=20, edgecolor='black', color='red', alpha=0.7)
axes[1].axvline(np.mean(rho_values), color='blue', linestyle='--', linewidth=2, label=f'Mean: {np.mean(rho_values):.4f}')
axes[1].set_xlabel('ρ (1/day)', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Distribution of Proliferation Rates', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Scatter plot D vs ρ
axes[2].scatter(D_values, rho_values, s=100, alpha=0.6, edgecolor='black')
axes[2].set_xlabel('D (mm²/day)', fontsize=12)
axes[2].set_ylabel('ρ (1/day)', fontsize=12)
axes[2].set_title('D vs ρ Correlation', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3)

# Compute correlation
corr = np.corrcoef(D_values, rho_values)[0, 1]
axes[2].text(0.05, 0.95, f'Correlation: {corr:.3f}', 
            transform=axes[2].transAxes, fontsize=12,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('phase_a_parameter_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

### 8.2 Save Phase A Results

In [ ]:
# Save all parameters to CSV
params_df = pd.DataFrame([
    {
        'patient_id': pid,
        'D': result['params']['D'],
        'rho': result['params']['rho']
    }
    for pid, result in phase_a_results.items()
])

params_df.to_csv('phase_a_learned_parameters.csv', index=False)
print("\n✓ Parameters saved to: phase_a_learned_parameters.csv")

# Display parameter table
print("\nFirst 10 patients:")
print(params_df.head(10))

## 9. Visualization: Tumor Growth Prediction

### 9.1 Generate Tumor Evolution Sequence

In [ ]:
def predict_tumor_evolution(model, volume_shape, n_timepoints=5, T_max=1.0):
    """
    Predict tumor evolution over time using trained PINN
    
    Args:
        model: Trained PINN model
        volume_shape: Shape of 3D volume (depth, height, width)
        n_timepoints: Number of time points to generate
        T_max: Maximum time (normalized)
    
    Returns:
        evolution: (n_timepoints, depth, height, width) array
    """
    model.eval()
    depth, height, width = volume_shape
    
    # Create spatial grid
    z = np.linspace(0, 1, depth)
    y = np.linspace(0, 1, height)
    x = np.linspace(0, 1, width)
    
    Z, Y, X = np.meshgrid(z, y, x, indexing='ij')
    
    evolution = []
    
    with torch.no_grad():
        for t_val in tqdm(np.linspace(0, T_max, n_timepoints), desc='Generating timepoints'):
            # Create input points: [t, x, y, z]
            T = np.full_like(X, t_val)
            points = np.stack([T.flatten(), X.flatten(), Y.flatten(), Z.flatten()], axis=1)
            points_tensor = torch.from_numpy(points.astype(np.float32)).to(device)
            
            # Predict tumor density
            u_pred = model(points_tensor).cpu().numpy().reshape(volume_shape)
            evolution.append(u_pred)
    
    return np.array(evolution)


# Generate evolution for test patient
test_patient_result = phase_a_results[test_patient_id]
test_volume_shape = (30, 64, 64)  # Example shape

print(f"Generating tumor evolution for patient: {test_patient_id}")
evolution = predict_tumor_evolution(
    test_patient_result['model'],
    test_volume_shape,
    n_timepoints=5,
    T_max=1.0
)

print(f"Evolution shape: {evolution.shape}")

In [ ]:
def visualize_tumor_evolution(evolution, patient_id):
    """
    Visualize tumor growth over time
    """
    n_timepoints, depth, height, width = evolution.shape
    
    # Select middle slice
    mid_slice = depth // 2
    
    fig, axes = plt.subplots(1, n_timepoints, figsize=(4*n_timepoints, 4))
    
    for i in range(n_timepoints):
        im = axes[i].imshow(evolution[i, mid_slice], cmap='hot', vmin=0, vmax=1)
        axes[i].set_title(f't = {i/(n_timepoints-1):.2f}', fontsize=14, fontweight='bold')
        axes[i].axis('off')
        plt.colorbar(im, ax=axes[i], fraction=0.046)
    
    plt.suptitle(f'Tumor Evolution - Patient: {patient_id} (Slice {mid_slice})', 
                fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.savefig(f'tumor_evolution_{patient_id}.png', dpi=300, bbox_inches='tight')
    plt.show()


# Visualize evolution
visualize_tumor_evolution(evolution, test_patient_id)

## 10. Model Summary and Export

In [ ]:
print("\n" + "="*70)
print("PINN TUMOR GROWTH PREDICTION - MODEL SUMMARY")
print("="*70)
print(f"\nArchitecture: Physics-Informed Neural Network")
print(f"PDE: Fisher-Kolmogorov (∂u/∂t = D∇²u + ρu(1-u))")
print(f"Network: [4, 20, 50, 50, 50, 50, 20, 1]")
print(f"Total Parameters: {total_params:,}")

print(f"\n--- Phase A: Parameter Estimation ---")
print(f"Dataset: LGG (110 patients, baseline segmentations)")
print(f"Patients trained: {len(phase_a_results)}")
print(f"Training epochs per patient: {PHASE_A_EPOCHS}")

if len(D_values) > 0:
    print(f"\nLearned Parameters (Mean ± Std):")
    print(f"  D (diffusion): {np.mean(D_values):.6f} ± {np.std(D_values):.6f} mm²/day")
    print(f"  ρ (proliferation): {np.mean(rho_values):.6f} ± {np.std(rho_values):.6f} 1/day")

print(f"\n--- Phase B: Growth Prediction (TODO) ---")
print(f"Dataset: MU-Glioma (203 patients, longitudinal)")
print(f"Goal: Validate forward predictions on follow-up scans")
print(f"Metrics: DICE score, tumor volume accuracy")

print(f"\n--- Outputs ---")
print(f"✓ Patient-specific physics parameters")
print(f"✓ Tumor evolution predictions (t = 0 → T)")
print(f"✓ Uncertainty quantification (ensemble)")
print(f"✓ Parameter distribution analysis")

print("\n" + "="*70)
print("TRAINING COMPLETE!")
print("="*70)
print(f"\nNext steps:")
print(f"1. Train on all {len(lgg_patient_data)} LGG patients (increase epochs to 5000-10000)")
print(f"2. Implement Phase B with MU-Glioma longitudinal data")
print(f"3. Train ensemble of PINNs for uncertainty quantification")
print(f"4. Validate DICE scores on follow-up scans")
print(f"5. Clinical interpretation of learned parameters")
print("="*70)